# Campus Maintenance AI: Embeddings and Feature Selection

| Activity | Implementation / learning goal | Exact slides in the supplied Part B deck | Suggested time |
|---|---|---|---:|
| Setup | Colab installation, imports and how to use this notebook | 2: Topics Covered | 5 min |
| Exercise 1 | Inspect learned word embeddings, cosine similarity and a 2D projection | 5: Encoding Text Data; 10: Word Embeddings; 12: Encoding Text Data | 6 min |
| Exercise 2 | Train Word2Vec and average word vectors to compare maintenance messages | 3–4: Text preprocessing; 10: Word Embeddings; 11: Representing a Sentence with Word2Vec; 12: Encoding Text Data | 10 min |
| Exercise 3 | Rank numerical features using ANOVA and select the top three | 20: Filter Methods; 23: Analysis of Variance; 24: ANOVA; 25: ANOVA F-score worked example | 7 min |
| Exercise 4 | Forward selection: add the feature that improves training-set CV accuracy most | 27: Wrapper Methods; 28: Forward Selection; 29: Forward Feature Selection | 7 min |
| Exercise 5 | Backward selection: test removals and stop when the improvement is too small | 27: Wrapper Methods; 30: Backward Selection; 31: Backward Feature Selection | 6 min |
| Exercise 6 | Recursive feature elimination using scaled logistic-regression coefficients | 32: Recursive Feature Elimination | 6 min |
| Exercise 7 | Embedded selection with Lasso for a continuous energy-use target | 18: Method families; 35: Embedded Methods | 8 min |
| Finish | One final held-out classification evaluation and reflection | 17: Why accuracy reduces; 36: Choosing a method | 5 min |
| Optional 1 | Compare BoW and embeddings; investigate word order and unseen words | 5–12: Text representations, especially 8 and 11 | Extra 10 min |
| Optional 2 | RFECV and the stability of selected features under different CV splits | 17–19: Noise and dimensionality; 32: RFE / RFECV note; 36: Choosing a method | Extra 10–15 min |
| Optional 3 | Tune Lasso and Ridge using CV and compare their coefficients | 35: Embedded Methods; 36: Choosing a method | Extra 10–15 min |

**Core total: 60 minutes.** Optional exercises are outside that hour. Slide numbers are the physical positions 1–37 in `Week04_Lecture01_Feature_Engineering_PartB_ForChatGPT.pptx`, including its title slide. Slide 25 is the image-only ANOVA worked example.

## Student notebook

A university facilities team wants to understand maintenance requests and use
sensor readings to anticipate heating-system faults. Later, the same sensor
features support a separate energy-use regression task. All data are synthetic
teaching examples, not university records or a validated engineering system.

**In Colab:** open [Google Colab](https://colab.research.google.com/), choose
**File → Upload notebook**, and upload this `.ipynb` file. Use a CPU runtime.
Run setup, then work through Exercises 1–7. Each task has a small `TODO` and
questions to answer in its following markdown cell. Provided code handles the
longer loops so that the hour focuses on understanding and small implementations.

`None` marks an unfinished answer. These cells print guidance and skip their
output instead of crashing. Later core exercises have independent inputs, so
you can move on if stuck. Optional exercises default to OFF.

## Setup · 5 minutes

Colab normally provides NumPy, pandas, Matplotlib and scikit-learn. The next cell
installs Gensim and ensures the minimum versions used here. Run it **before**
imports. A fresh runtime is recommended if you previously imported another
Gensim version. After installation, no network access or additional files are needed.

In [ ]:
%pip -q install "gensim==4.4.0" "scikit-learn>=1.5,<2" "pandas>=2,<3" "matplotlib>=3.7,<4"

In [ ]:
import re
import zlib
import time
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, f_classif, RFE, RFECV
from sklearn.linear_model import LogisticRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     KFold, cross_val_score, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
NAVY, BLUE, TEAL, ORANGE = '#102a43', '#2878b5', '#159895', '#de8f32'
plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.labelcolor': NAVY, 'axes.titlecolor': NAVY,
                     'figure.dpi': 110})
for package in ['gensim', 'numpy', 'pandas', 'scikit-learn']:
    print(f'{package}: {importlib.metadata.version(package)}')
print('Ready. CPU is sufficient; no external datasets or large models are downloaded.')

## Exercise 1 · Inspect learned word embeddings · 6 minutes

**Slides 5, 10 and 12.** An embedding is a dense vector, not a word count.
The provided code trains a tiny Word2Vec model on maintenance language so you
can inspect genuine learned vectors without downloading a large model.
The controlled corpus substitutes device names into heating, plumbing and
lighting contexts. Repeating its 36 unique messages gives more training
exposures, **not** new knowledge. Do not treat it as a general language model.

1. Run the provided training cell and inspect two corpus sentences.
2. Complete the vector lookup for `radiator` using `starter_model.wv[word]`.
3. Compare cosine similarities and inspect the 2D projection.

Cosine similarity compares directions: $\cos(u,v)=\frac{u^Tv}{\|u\|\|v\|}$.
It is not a probability. The full vectors here have 24 dimensions.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

def stable_hash(text):
    return zlib.adler32(text.encode("utf-8"))

def make_corpus():
    themes = {
        "heating": {
            "words": ["heater", "radiator", "boiler"],
            "templates": [
                "the {word} keeps the room warm during winter",
                "repair the {word} because the room feels cold",
                "the {word} provides heating for the cold office",
                "check the {word} heating temperature in the room",
            ],
        },
        "plumbing": {
            "words": ["tap", "faucet", "pipe"],
            "templates": [
                "the {word} leaks water near the sink",
                "repair the {word} because water drips onto the floor",
                "the {word} supplies water to the bathroom sink",
                "check the {word} water leak in the bathroom",
            ],
        },
        "lighting": {
            "words": ["lamp", "light", "bulb"],
            "templates": [
                "the {word} makes the dark corridor bright",
                "repair the {word} because the corridor looks dark",
                "the {word} provides bright lighting for the hallway",
                "check the {word} electrical lighting in the corridor",
            ],
        },
    }
    sentences = [tokenize(template.format(word=word))
                 for item in themes.values()
                 for word in item["words"] for template in item["templates"]]
    rng = np.random.default_rng(SEED)
    corpus = [sentences[i].copy() for i in rng.permutation(np.tile(np.arange(len(sentences)), 20))]
    return themes, corpus

def train_words(corpus, vector_size=24, window=3):
    return Word2Vec(sentences=corpus, vector_size=vector_size, window=window,
                    min_count=1, sg=1, epochs=40, workers=1, seed=SEED,
                    sample=0, hashfxn=stable_hash)



themes, corpus = make_corpus()
starter_model = train_words(corpus)
print(f'{len(corpus)} training sentences; {len(starter_model.wv)} vocabulary words')
print('First two tokenised sentences:', corpus[:2])

In [ ]:
word = 'radiator'
embedding = None  # TODO: look up this word in starter_model.wv
if embedding is None:
    print('Complete the vector lookup, then rerun this cell.')
else:
    assert embedding.shape == (24,) and np.isfinite(embedding).all()
    print('Shape:', embedding.shape)
    display(pd.Series(embedding[:8], index=[f'dim_{i}' for i in range(8)]))
    print('heater / radiator cosine:', starter_model.wv.similarity('heater', 'radiator'))
    print('heater / faucet cosine:  ', starter_model.wv.similarity('heater', 'faucet'))
    display(pd.DataFrame(starter_model.wv.most_similar(word, topn=5),
                         columns=['neighbour', 'cosine_similarity']))

In [ ]:
# Coordinates come from the learned vectors; colours identify corpus themes.
fig, ax = plt.subplots(figsize=(8, 4))
words = [word for item in themes.values() for word in item['words']]
xy = PCA(n_components=2).fit_transform(starter_model.wv[words])
offset = 0
for (theme, item), colour in zip(themes.items(), [ORANGE, BLUE, TEAL]):
    n = len(item['words'])
    ax.scatter(xy[offset:offset+n, 0], xy[offset:offset+n, 1],
               color=colour, label=theme, s=60)
    for j, word in enumerate(item['words']):
        label_offset = [(-38, 20), (-12, -25), (23, 8)][j]
        ax.annotate(word, xy[offset+j], xytext=label_offset,
                    textcoords='offset points', fontsize=10,
                    arrowprops={'arrowstyle': '-', 'color': colour, 'lw': 0.8})
    offset += n
ax.set(xlabel='Principal component 1', ylabel='Principal component 2',
       title='Learned 24D vectors projected into 2D')
ax.margins(0.2); ax.legend(loc='best'); ax.grid(alpha=0.15)
plt.tight_layout(); plt.show()

**Record:** What is the vector length? Which pair has higher cosine similarity?
Does the label “dimension 3” have a fixed human meaning? Can a 2D projection
preserve every relationship in the 24D vectors?

**Your explanation:** _Write here._

## Exercise 2 · Train Word2Vec and compare messages · 10 minutes

**Slides 3–4 and 10–12.** Train your own skip-gram Word2Vec model. `vector_size`
sets vector length, `window` controls context range, `min_count=1` keeps rare
tokens, and `sg=1` chooses skip-gram. Gensim handles training internally.

1. Run the training recipe below. It creates a new model, not a copy of vectors.
2. Complete **one line** in `sentence_embedding`: stack known word vectors and
   average along `axis=0`. Hint: `word_model.wv[known]` has shape
   `(number_of_known_words, vector_size)`.
3. Rank the three candidate requests by similarity to the query.

$$s = \frac{1}{m}\sum_{i=1}^m v(w_i)$$

Here $m$ counts **known token occurrences**, including repeated words. Unknown
words are reported and ignored. If no tokens are known, return `None` rather
than invent a meaningful zero vector. Averaging preserves vector length and
loses word order. Word2Vec itself gives one fixed vector per vocabulary word.

In [ ]:
word_model = Word2Vec(
    sentences=corpus, vector_size=24, window=3, min_count=1,
    sg=1, epochs=40, workers=1, seed=SEED, sample=0, hashfxn=stable_hash
)
print('Trained vector length:', word_model.vector_size)

In [ ]:
def sentence_embedding(text):
    tokens = tokenize(text)
    known = [token for token in tokens if token in word_model.wv]
    if not known:
        return None
    return None  # TODO: return the mean vector across known token occurrences

query = 'radiator warm office'
requests = ['boiler heating temperature room',
            'tap water sink bathroom',
            'lamp bright corridor lighting']
query_vector = sentence_embedding(query)
if query_vector is None:
    print('Complete the mean-vector line above, then rerun this cell.')
else:
    request_vectors = np.vstack([sentence_embedding(text) for text in requests])
    similarities = cosine_similarity(query_vector.reshape(1, -1), request_vectors)[0]
    ranking = pd.DataFrame({'request': requests, 'cosine_similarity': similarities})
    display(ranking.sort_values('cosine_similarity', ascending=False))
    print('Sentence vector shape:', query_vector.shape)
    unknown = [t for t in tokenize('radiator xyzzy') if t not in word_model.wv]
    print('Unknown tokens in the second test:', unknown)
    assert sentence_embedding('xyzzy') is None
    assert query_vector.shape == (word_model.vector_size,)
    fig, ax = plt.subplots(figsize=(8, 3))
    order = ranking.sort_values('cosine_similarity')
    ax.barh(order['request'], order['cosine_similarity'], color=TEAL)
    ax.set(xlabel='Cosine similarity to query', title=f'Query: {query}')
    ax.set_xlim(-1, 1); plt.tight_layout(); plt.show()

**Record:** Which request is the closest match? Why is the sentence vector 24D
regardless of sentence length? What happens to `radiator xyzzy` and to `xyzzy`?

**Your explanation:** _Write here._

## Shared sensor data for Exercises 3–7

The facilities team now studies **360 synthetic equipment observations** with
eight numerical predictors. For Exercises 3–6 the target `fault` is categorical:
0 = no fault, 1 = fault. Readings represent measurements available before the
outcome, not fields recorded after a repair. This is a simulated snapshot task,
so we use a stratified random split rather than a time-series split.

| Predictor | Meaning / unit |
|---|---|
| room_temp_c | Room temperature, °C |
| pipe_pressure_bar | Heating-pipe pressure, bar |
| vibration_mm_s | Equipment vibration, mm/s |
| humidity_pct | Relative humidity, % |
| outdoor_temp_c | Outdoor temperature, °C |
| floor_area_m2 | Room floor area, m² |
| asset_age_years | Equipment age, years |
| background_noise_db | Background sound level, dB |

Split once: **270 training rows, 90 held-out test rows**. All selection and
scaling use training data. Three-fold CV within training uses 180 rows to fit
and 90 rows to validate per fold. Scalers are fitted separately in each fold.
Keep the test set closed until the final audit.

In [ ]:
def make_sensor_data():
    rng = np.random.default_rng(SEED)
    z = rng.normal(size=(360, 8))
    X = pd.DataFrame({
        "room_temp_c": 21 - 2 * z[:, 0],
        "pipe_pressure_bar": 3 - 0.35 * z[:, 1],
        "vibration_mm_s": 2 + 0.5 * z[:, 2],
        "humidity_pct": 50 + 8 * z[:, 3],
        "outdoor_temp_c": 12 + 5 * z[:, 4],
        "floor_area_m2": 80 + 12 * z[:, 5],
        "asset_age_years": 10 + 2 * z[:, 6],
        "background_noise_db": 40 + 5 * z[:, 7],
    })
    risk = 1.9 * z[:, 0] + 1.5 * z[:, 1] + 1.1 * z[:, 2]
    y = pd.Series((risk + rng.normal(scale=1.3, size=len(X)) > 0).astype(int), name="fault")
    energy = pd.Series(30 + 5 * z[:, 0] + 3 * z[:, 1] + 2 * z[:, 2]
                       + rng.normal(scale=2, size=len(X)), name="energy_kwh")
    return X, y, energy

def make_clf():
    return Pipeline([("scale", StandardScaler()),
                     ("classifier", LogisticRegression(C=1, max_iter=2000, random_state=SEED))])



X, y, energy = make_sensor_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
energy_train = energy.loc[X_train.index]  # A separate continuous target for E7.
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
print('Training shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Training class counts:', y_train.value_counts().sort_index().to_dict())
display(X_train.head().round(2))
baseline_scores = cross_val_score(make_clf(), X_train, y_train, cv=CV,
                                  scoring='accuracy', n_jobs=1)
print(f'All-feature baseline CV accuracy: {baseline_scores.mean():.1%}')

**About the simulation:** the data-generation code intentionally contains signal
and irrelevant predictors. Known generating relationships are useful for
checking a teaching example; they would not be known in real data. A selected
feature is not automatically causal, and no method guarantees a better score.

## Exercise 3 · ANOVA filter selection · 7 minutes

**Slides 20 and 23–25.** Use `f_classif` because inputs are numerical and the
target is a class label. It compares class means one feature at a time:

$$F = \frac{\text{between-class mean square}}{\text{within-class mean square}}.$$

A larger F-score indicates stronger mean separation relative to within-class
spread. ANOVA **measures** these differences; it does not alter class variance.
`k=3` below is a teaching budget, not a claim that three is always optimal.

1. Run the training-only F-score plot. Compare a strong feature with a weak one.
2. Set the scoring function in the task to `f_classif` and fit the pipeline.
3. Inspect its selected names and cross-validation score.

In [ ]:
f_values, p_values = f_classif(X_train, y_train)
anova_table = pd.DataFrame({'feature': X_train.columns, 'F_score': f_values,
                            'p_value': p_values}).sort_values('F_score', ascending=False)
display(anova_table.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.7))
ordered = anova_table.sort_values('F_score')
axes[0].barh(ordered['feature'], ordered['F_score'], color=BLUE)
axes[0].set(xlabel='ANOVA F-score', title='Rank one feature at a time')
for label, feature, colour in [(0, 'room_temp_c', TEAL), (1, 'humidity_pct', ORANGE)]:
    # Standardise only for a shared visual scale, not for the ANOVA calculation.
    values = (X_train[feature] - X_train[feature].mean()) / X_train[feature].std()
    x_positions = np.arange(2) + label * 3
    boxes = axes[1].boxplot([values[y_train == cls] for cls in [0, 1]],
                           positions=x_positions, widths=0.65, patch_artist=True)
    for box in boxes['boxes']: box.set_facecolor(colour)
axes[1].set_xticks([0, 1, 3, 4], ['No fault', 'Fault', 'No fault', 'Fault'])
axes[1].set(ylabel='Feature value (training z-score)',
            title='Room temperature (teal) vs humidity (orange)')
plt.tight_layout(); plt.show()

In [ ]:
score_function = None  # TODO: choose the ANOVA classification function
anova_pipeline = None
if score_function is None:
    print('Choose the score function and rerun this cell.')
else:
    anova_pipeline = Pipeline([
        ('select', SelectKBest(score_func=score_function, k=3)),
        ('scale', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=2000, random_state=SEED))
    ])
    # Selection happens anew INSIDE each training fold, before fold validation.
    scores = cross_val_score(anova_pipeline, X_train, y_train, cv=CV,
                              scoring='accuracy', n_jobs=1)
    anova_pipeline.fit(X_train, y_train)
    names = X_train.columns[anova_pipeline['select'].get_support()].tolist()
    print('Selected:', names)
    print(f'Pipeline CV accuracy: {scores.mean():.1%}')

**Record:** Which three features have the highest F-scores? Why use `f_classif`
rather than `f_regression`? Could two highly correlated predictors both rank
highly? Does a tiny p-value prove a causal relationship?

**Your explanation:** _Write here._

## Provided wrapper-search helper

Read the comments, then run this cell once. Each candidate subset uses the same
classifier, metric and CV splits. **Forward** tests each possible addition.
**Backward** tests each possible removal, including a comparison with the full
model before accepting the first removal. Both stop when the best candidate
improves by no more than **0.5 percentage point** (`min_gain=0.005`). The final
rejected proposal appears in the trace but is excluded from the chosen subset.

These are greedy searches, so they do not guarantee the globally best subset.
Their repeatedly consulted **search CV scores can be optimistic**. They are
search guidance, not independent final performance estimates. A proper unbiased
comparison of entire search procedures needs outer validation / nested CV.

In [ ]:
# Provided helper: each candidate gets a new scaler and classifier in EACH CV fold.
# The cache avoids repeating identical subset evaluations on the same fixed CV splits.
_subset_cache = {}

def subset_cv_accuracy(features):
    key = tuple(c for c in X_train.columns if c in features)
    if not key:
        raise ValueError('Use the dummy baseline for an empty feature set.')
    if key not in _subset_cache:
        scores = cross_val_score(make_clf(), X_train.loc[:, list(key)], y_train,
                                 cv=CV, scoring='accuracy', n_jobs=1)
        _subset_cache[key] = float(scores.mean())
    return _subset_cache[key]

def greedy_search(direction, min_gain=0.005):
    """Return chosen features, the best action per round, and every candidate score.

    min_gain=0.005 means require an improvement of MORE THAN 0.5 percentage point.
    This is a teaching stopping rule, not a statistical significance test.
    Ties are resolved by the original column order.
    """
    if direction not in ('forward', 'backward'):
        raise ValueError("direction must be 'forward' or 'backward'")
    selected = [] if direction == 'forward' else list(X_train.columns)
    if direction == 'forward':
        current = float(cross_val_score(DummyClassifier(strategy='most_frequent'),
                        X_train, y_train, cv=CV, scoring='accuracy').mean())
    else:
        current = subset_cv_accuracy(selected)
    history = [dict(step=0, action='start', n_features=len(selected),
                    cv_accuracy=current, accepted=True, features=tuple(selected))]
    candidates = []
    while True:
        choices = ([c for c in X_train.columns if c not in selected]
                   if direction == 'forward' else selected.copy())
        if not choices or (direction == 'backward' and len(selected) == 1):
            break  # Always retain at least one predictor in backward search.
        round_rows = []
        for feature in choices:
            subset = (selected + [feature] if direction == 'forward'
                      else [c for c in selected if c != feature])
            round_rows.append(dict(step=len(history), action=feature,
                                   n_features=len(subset),
                                   cv_accuracy=subset_cv_accuracy(subset),
                                   features=tuple(subset)))
        candidates.extend(round_rows)
        best = max(round_rows, key=lambda row: row['cv_accuracy'])
        accepted = bool(best['cv_accuracy'] > current + min_gain + 1e-12)
        history.append(dict(best, accepted=accepted))
        if not accepted:
            break  # Do NOT adopt the rejected candidate.
        selected = list(best['features'])
        current = best['cv_accuracy']
    return {'selected': selected, 'history': pd.DataFrame(history),
            'candidates': pd.DataFrame(candidates), 'cv_accuracy': current}

def show_search(result, title):
    history = result['history'].copy()
    history['CV accuracy (%)'] = 100 * history['cv_accuracy']
    display(history[['step', 'action', 'n_features', 'CV accuracy (%)', 'accepted']])
    print('Final selected features:', result['selected'])
    fig, ax = plt.subplots(figsize=(8, 3.4))
    accepted = history[history['accepted']]
    ax.plot(accepted['step'], 100 * accepted['cv_accuracy'], 'o-', color=TEAL,
            label='Accepted steps')
    rejected = history[~history['accepted']]
    if len(rejected):
        ax.scatter(rejected['step'], 100 * rejected['cv_accuracy'], marker='x',
                   s=110, linewidths=2, color=ORANGE, label='Rejected next step')
    ax.set(xlabel='Search step', ylabel='Search CV accuracy (%)', title=title)
    ax.legend(); ax.grid(axis='y', alpha=0.18)
    plt.tight_layout(); plt.show()

## Exercise 4 · Forward selection · 7 minutes

**Slides 27–29.** First inspect every single-feature model, then run the greedy
search. Complete `direction` with the appropriate string. After the first
accepted feature, all later candidates retain previously accepted features.
The zero-feature starting point is a most-frequent-class dummy classifier.

In [ ]:
single_feature_scores = pd.Series({feature: subset_cv_accuracy([feature])
                                   for feature in X_train.columns}).sort_values()
ax = (100 * single_feature_scores).plot.barh(color=BLUE, figsize=(8, 3.5))
ax.set(xlabel='Training-set CV accuracy (%)', title='Forward search: first candidates')
plt.tight_layout(); plt.show()

In [ ]:
direction = None  # TODO: 'forward' or 'backward'?
forward_result = None
if direction is None:
    print('Set the search direction, then rerun.')
else:
    forward_result = greedy_search(direction=direction, min_gain=0.005)
    show_search(forward_result, 'Forward selection: accepted and rejected steps')
    print('Last-round candidate scores:')
    last = forward_result['candidates']['step'].max()
    display(forward_result['candidates'].query('step == @last')
            [['action', 'cv_accuracy']].sort_values('cv_accuracy', ascending=False))

**Record:** Which feature enters first? Which features remain at the stopping
point? Why is the rejected proposal excluded? Inspect the helper and identify
the line that constructs a candidate by adding one feature.

**Your explanation:** _Write here._

## Exercise 5 · Backward selection · 6 minutes

**Slides 27 and 30–31.** Start with all eight features. Complete the direction
and inspect which removal improves CV accuracy most. A feature that survived
forward search is not required to survive backward search, or vice versa.

In [ ]:
direction = None  # TODO: choose the backward-search direction
backward_result = None
if direction is None:
    print('Set the search direction, then rerun.')
else:
    backward_result = greedy_search(direction=direction, min_gain=0.005)
    show_search(backward_result, 'Backward selection: accepted and rejected removals')
    first_round = backward_result['candidates'].query('step == 1')
    display(first_round[['action', 'cv_accuracy']].sort_values('cv_accuracy', ascending=False))
    if forward_result is not None:
        print('Shared features:', sorted(set(forward_result['selected']) &
                                         set(backward_result['selected'])))

**Record:** Which feature is removed first? How many remain? Why can the two
greedy directions disagree? Does keeping a feature prove it truly matters?

**Quick extension if time permits:** rerun either search with `min_gain=0.0`.
Compare its chosen subset with the 0.005 rule. Keep the test set closed.

**Your explanation:** _Write here._

## Exercise 6 · Recursive feature elimination · 6 minutes

**Slide 32.** RFE refits a model after each elimination and removes the predictor
with the smallest absolute coefficient. It **does not** fit and score every
possible removal like backward selection. We fix a three-feature budget in
advance for this exercise. RFE alone does not choose that budget by CV.

Set `n_keep=3`. The estimator is a pipeline containing StandardScaler and
logistic regression, so importances use comparable predictor scales. The
`importance_getter` locates that fitted classifier's coefficients. When RFE
is evaluated by CV, its entire elimination process happens within each fold.

In [ ]:
n_keep = None  # TODO: retain three features
rfe_selector = None
if n_keep is None:
    print('Set the target number of features, then rerun.')
else:
    rfe_selector = RFE(estimator=make_clf(), n_features_to_select=n_keep, step=1,
                       importance_getter='named_steps.classifier.coef_')
    scores = cross_val_score(rfe_selector, X_train, y_train, cv=CV,
                              scoring='accuracy', n_jobs=1)
    rfe_selector.fit(X_train, y_train)
    rfe_table = pd.DataFrame({'feature': X_train.columns,
                              'selected': rfe_selector.support_,
                              'rank': rfe_selector.ranking_}).sort_values('rank')
    display(rfe_table)
    print('Selected:', X_train.columns[rfe_selector.support_].tolist())
    print(f'RFE pipeline CV accuracy: {scores.mean():.1%}')
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.barh(rfe_table['feature'][::-1], rfe_table['rank'][::-1],
             color=[TEAL if flag else '#b7c9d6' for flag in rfe_table['selected'][::-1]])
    ax.set(xlabel='RFE rank (1 = retained)', title='RFE ranking; selected features tie at rank 1')
    plt.tight_layout(); plt.show()

**Record:** Which predictors have rank 1? Does rank 1 order those retained
features by importance? What determines a removal in RFE versus backward
selection? Why scale before comparing linear-model coefficients?

**Your explanation:** _Write here._

## Exercise 7 · Embedded selection using Lasso · 8 minutes

**Slides 18 and 35.** We now switch to the separate **continuous** target
`energy_kwh`, simulated energy use in kWh. Use Lasso regression for this target,
not as a probability model for the binary fault label. The training/test row
split stays the same; this exercise uses only the training rows.

Lasso learns coefficients and can set some exactly to zero during fitting:

$$\underset{\beta}{\operatorname{minimise}}\quad
\frac{1}{2n}\|y-X\beta\|_2^2+\alpha\sum_j|\beta_j|.$$

The model also fits an unpenalised intercept. Standardise predictors first.
Larger `alpha` applies stronger shrinkage; making it too large can underfit.
Here 0.5 is a teaching setting, not an automatically optimal value.

1. Run the coefficient path, which uses training data only.
2. Complete the estimator in the pipeline with `Lasso(alpha=0.5, max_iter=10000)`.
3. Identify selected features using nonzero coefficients. Then try 0.05 and 6.0.

**Distinction:** Ridge usually shrinks coefficients without making them exactly
zero, so Ridge alone is generally not a method for selecting a feature subset.

In [ ]:
alphas = np.logspace(-2, 1, 35)
coefficient_path = []
for alpha in alphas:
    fitted = Pipeline([('scale', StandardScaler()),
                       ('lasso', Lasso(alpha=alpha, max_iter=10000))])
    fitted.fit(X_train, energy_train)
    coefficient_path.append(fitted['lasso'].coef_)
coefficient_path = np.asarray(coefficient_path)
fig, ax = plt.subplots(figsize=(9, 4))
for j, feature in enumerate(X_train.columns):
    ax.plot(alphas, coefficient_path[:, j], label=feature)
ax.set_xscale('log'); ax.axhline(0, color='grey', linewidth=0.8)
ax.set(xlabel='alpha (log scale)', ylabel='Coefficient per training standard deviation',
        title='Lasso drives coefficients towards zero')
ax.legend(fontsize=8, loc='upper right', ncol=2); ax.grid(alpha=0.15)
plt.tight_layout(); plt.show()

In [ ]:
lasso_estimator = None  # TODO: construct Lasso with alpha=0.5
lasso_pipeline = None
if lasso_estimator is None:
    print('Create the Lasso estimator, then rerun this cell.')
else:
    lasso_pipeline = Pipeline([('scale', StandardScaler()), ('lasso', lasso_estimator)])
    lasso_pipeline.fit(X_train, energy_train)
    coefficients = pd.Series(lasso_pipeline['lasso'].coef_, index=X_train.columns)
    selected = coefficients.abs() > 1e-8  # Numerical tolerance around zero.
    display(pd.DataFrame({'coefficient': coefficients, 'selected': selected}))
    print('Retained:', coefficients.index[selected].tolist())
    ax = coefficients.sort_values().plot.barh(color=TEAL, figsize=(8, 3.5))
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set(xlabel='Coefficient per training standard deviation', title='Lasso embedded selection')
    plt.tight_layout(); plt.show()

**Record:** How many coefficients remain nonzero at alpha 0.05, 0.5 and 6.0?
Why does zero imply exclusion from the fitted prediction? Why can very large
alpha be harmful? Why should we not expect the same features as a model for a
different target, such as fault classification?

**Your explanation:** _Write here._

## Finish · One final evaluation and reflection · 5 minutes

Complete E6 first. We predeclared **RFE with three features** as the model for
the held-out classification audit. The next cell uses the 90 test observations
once, after fitting on all 270 training observations. Do not use its result to
change the budget, stopping rules or method. If you change those choices after
seeing the result, this test set no longer provides an independent final check.

The search scores in E4–E5 helped select subsets and can be optimistic. Even
ordinary CV has sampling uncertainty, and the smaller test set can disagree
with its estimate. No selected model is guaranteed to improve performance.

In [ ]:
if rfe_selector is None:
    print('Complete Exercise 6 to unlock the final held-out evaluation.')
else:
    test_predictions = rfe_selector.predict(X_test)
    test_accuracy = accuracy_score(y_test, test_predictions)
    print(f'Final held-out accuracy: {test_accuracy:.1%} ({len(y_test)} test rows)')
    print('Features:', X_train.columns[rfe_selector.support_].tolist())
    ConfusionMatrixDisplay.from_predictions(
        y_test, test_predictions, display_labels=['No fault', 'Fault'],
        cmap='Blues', colorbar=False)
    plt.title('One final test: predeclared RFE model')
    plt.tight_layout(); plt.show()

**Exit questions:**

1. Which method is a filter, which methods are wrappers, and which is embedded?
2. Why can forward selection, backward selection and RFE retain different subsets?
3. How does an embedding coordinate differ from a selected original sensor feature?
4. Why must selection and scaling stay inside the appropriate training split?

**Your explanation:** _Write here._

# Optional exercises · after completing the core hour

These use training data or the existing toy corpus. Leave the held-out test set
closed. Turn on only the exercise you want by setting its flag to `True`.

## Optional 1 · Representation limits and retrieval · 10 minutes

**Slides 5–12, especially 8 and 11.** Complete E2 first.

Compare a BoW query vector with an averaged Word2Vec query vector for the same
candidate requests. Then compare `the heater keeps the room warm` with
`the room keeps the heater warm`:
they contain the same token multiset but suggest different roles.

**Tasks:** enable the cell; interpret both rankings; verify the two mean vectors
are equal; inspect the known-word coverage of `radiator xyzzy` and `xyzzy`.
Explain why neither simple representation resolves word order or all unseen words.

In [ ]:
RUN_OPTIONAL_1 = False

if RUN_OPTIONAL_1:
    if sentence_embedding(query) is None:
        print('Complete the mean-vector line in E2 first.')
    else:
        bow = CountVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
        bow.fit([' '.join(tokens) for tokens in corpus])
        bow_scores = cosine_similarity(bow.transform([query]), bow.transform(requests))[0]
        embedding_scores = cosine_similarity(sentence_embedding(query).reshape(1, -1),
                         np.vstack([sentence_embedding(text) for text in requests]))[0]
        display(pd.DataFrame({'request': requests, 'BoW cosine': bow_scores,
                               'mean-Word2Vec cosine': embedding_scores}))
        a, b = 'the heater keeps the room warm', 'the room keeps the heater warm'
        print('Mean vectors equal?', np.allclose(sentence_embedding(a), sentence_embedding(b)))
        print('BoW vectors equal?', np.array_equal(bow.transform([a]).toarray(),
                                                   bow.transform([b]).toarray()))
        for text in ['radiator xyzzy', 'xyzzy']:
            tokens = tokenize(text)
            known = [t for t in tokens if t in word_model.wv]
            print(text, '| coverage:', f'{len(known)}/{len(tokens)}',
                  '| mean vector available:', sentence_embedding(text) is not None)
else:
    print('Optional 1 is OFF.')

**Coding challenge:** construct a `CountVectorizer` with `ngram_range=(1, 2)`
and the same tokenizer. Fit it on the provided corpus and compare the reordered
messages again. Will including neighbouring word pairs distinguish them?
Complete the constructor below; the comparison code is provided.

In [ ]:
bigram_vectorizer = None  # TODO: CountVectorizer with unigrams and bigrams
if RUN_OPTIONAL_1:
    if bigram_vectorizer is None:
        print('Construct the unigram + bigram vectorizer to complete the coding challenge.')
    else:
        bigram_vectorizer.fit([' '.join(tokens) for tokens in corpus])
        order_pair = ['the heater keeps the room warm', 'the room keeps the heater warm']
        pair_vectors = bigram_vectorizer.transform(order_pair).toarray()
        print('Vectors equal with bigrams?', np.array_equal(pair_vectors[0], pair_vectors[1]))
        changed = pair_vectors[0] != pair_vectors[1]
        print('Differing features:', bigram_vectorizer.get_feature_names_out()[changed].tolist())

## Optional 2 · RFECV and selection stability · 10–15 minutes

**Slides 17–19, 32 and 36.** RFECV extends RFE by choosing the feature count with
training-set CV. The estimator again contains its own scaler. Run with CV seeds
42, 7 and 99 and compare selected sets. Different validation partitions can
produce different choices. Its peak search CV score is not an independent
final performance estimate; nested CV would assess the whole selection process.

**Tasks:** enable the cell; inspect the first CV curve; count how often each
feature is selected. Would one run be sufficient evidence of stable importance?

In [ ]:
RUN_OPTIONAL_2 = False

if RUN_OPTIONAL_2:
    stability_rows = []
    first_rfecv = None
    for cv_seed in [42, 7, 99]:
        selector = RFECV(estimator=make_clf(), step=1, min_features_to_select=1,
                         cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=cv_seed),
                         scoring='accuracy', n_jobs=1,
                         importance_getter='named_steps.classifier.coef_')
        selector.fit(X_train, y_train)
        if first_rfecv is None:
            first_rfecv = selector
        stability_rows.append(pd.Series(selector.support_.astype(int), index=X_train.columns,
                                         name=f'CV seed {cv_seed}'))
    stability = pd.DataFrame(stability_rows)
    display(stability)
    print('Selected counts by split:', stability.sum(axis=1).to_dict())
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    results = first_rfecv.cv_results_
    axes[0].errorbar(results['n_features'], 100 * results['mean_test_score'],
                     yerr=100 * results['std_test_score'], marker='o', color=BLUE, capsize=3)
    axes[0].set(xlabel='Number of retained features', ylabel='Search CV accuracy (%)',
                title='RFECV: mean accuracy ± fold SD')
    stability.sum().sort_values().plot.barh(ax=axes[1], color=TEAL)
    axes[1].set(xlabel='Number of runs selected (out of 3)', title='Selection stability')
    axes[1].set_xlim(0, 3.2)
    plt.tight_layout(); plt.show()
else:
    print('Optional 2 is OFF.')

**Coding challenge:** implement the Jaccard similarity of two selected feature
sets: $J(A,B)=|A\cap B|/|A\cup B|$. Return 1.0 when both sets are empty.
Fill the function below, then compare every pair of RFECV runs. A score of 1
means identical sets. How could a high overlap still include irrelevant features?

In [ ]:
def jaccard(a, b):
    return None  # TODO: intersection size divided by union size

if RUN_OPTIONAL_2:
    if jaccard({'x'}, {'x', 'y'}) is None:
        print('Complete the Jaccard function to compare selected sets.')
    else:
        assert jaccard(set(), set()) == 1.0
        feature_sets = [set(row.index[row.astype(bool)]) for _, row in stability.iterrows()]
        overlap = [[jaccard(a, b) for b in feature_sets] for a in feature_sets]
        display(pd.DataFrame(overlap, index=stability.index, columns=stability.index).round(3))

## Optional 3 · Tune Lasso and compare Ridge · 10–15 minutes

**Slides 35–36.** Use `GridSearchCV` around the **whole pipeline**, so scaling
is fitted within each regression CV fold. Compare Lasso with Ridge using a
continuous target and RMSE, not classification accuracy.

**Tasks:** enable the cell; compare the chosen alpha, CV RMSE and number of
nonzero coefficients. Explain why a Ridge coefficient plot is useful but is
generally not a selected subset. Think about what would be needed to claim
one tuned procedure generalises better without reusing its tuning scores.

In [ ]:
RUN_OPTIONAL_3 = False

if RUN_OPTIONAL_3:
    regression_cv = KFold(n_splits=3, shuffle=True, random_state=SEED)
    rows, comparison_coefficients = [], {}
    for name, estimator in [('Lasso', Lasso(max_iter=20000)), ('Ridge', Ridge())]:
        pipeline = Pipeline([('scale', StandardScaler()), ('model', estimator)])
        search = GridSearchCV(pipeline, {'model__alpha': np.logspace(-2, 1, 8)},
                              cv=regression_cv, scoring='neg_root_mean_squared_error', n_jobs=1)
        search.fit(X_train, energy_train)
        coefs = search.best_estimator_['model'].coef_
        comparison_coefficients[name] = coefs
        rows.append({'method': name, 'chosen_alpha': search.best_params_['model__alpha'],
                     'tuning_CV_RMSE_kwh': -search.best_score_,
                     'nonzero_coefficients': int(np.count_nonzero(np.abs(coefs) > 1e-8))})
    display(pd.DataFrame(rows).round(4))
    pd.DataFrame(comparison_coefficients, index=X_train.columns).plot.barh(
        figsize=(9, 4), color=[TEAL, ORANGE])
    plt.xlabel('Coefficient per training standard deviation')
    plt.title('Tuned Lasso vs Ridge: exact zeros versus shrinkage')
    plt.tight_layout(); plt.show()
else:
    print('Optional 3 is OFF.')

**Coding challenge:** add Elastic Net, which combines L1 and L2 penalties.
`l1_ratio=1` gives Lasso's penalty; values between 0 and 1 mix both penalties.
Construct an `ElasticNet` estimator below. The provided grid tunes both alpha
and l1_ratio using training-only CV. Compare its sparsity with Lasso and Ridge.

In [ ]:
elastic_estimator = None  # TODO: construct ElasticNet(max_iter=20000)
if RUN_OPTIONAL_3:
    if elastic_estimator is None:
        print('Construct ElasticNet to extend the embedded-method comparison.')
    else:
        elastic_pipeline = Pipeline([('scale', StandardScaler()), ('model', elastic_estimator)])
        elastic_search = GridSearchCV(elastic_pipeline,
            {'model__alpha': np.logspace(-2, 1, 8), 'model__l1_ratio': [0.2, 0.5, 0.8]},
            cv=KFold(n_splits=3, shuffle=True, random_state=SEED),
            scoring='neg_root_mean_squared_error', n_jobs=1)
        elastic_search.fit(X_train, energy_train)
        print('Elastic Net settings:', elastic_search.best_params_)
        print('Tuning CV RMSE (kWh):', round(-elastic_search.best_score_, 3))
        elastic_coefs = pd.Series(elastic_search.best_estimator_['model'].coef_, index=X_train.columns)
        display(pd.DataFrame({'coefficient': elastic_coefs,
                               'selected': elastic_coefs.abs() > 1e-8}))

## References and further reading

The exercise map refers to Dr Hamid Nasiri's supplied Week 4 Part B slides.
The corpus, sensor data and code examples in this notebook are new teaching
examples. Numerical outputs are computed by the notebook, not copied from slides.

- [Gensim: Word2Vec training, vector lookup and parameters](https://radimrehurek.com/gensim/models/word2vec.html)
- [scikit-learn: feature selection overview](https://scikit-learn.org/stable/modules/feature_selection.html)
- [scikit-learn: ANOVA F-values for classification](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html)
- [scikit-learn: recursive feature elimination](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html)
- [scikit-learn: Lasso objective and parameters](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
- [scikit-learn: Elastic Net](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)
- [scikit-learn: avoiding leakage with pipelines](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

**Reproducibility:** fixed NumPy seed, one Word2Vec worker and a stable hash
function reduce run-to-run variation. Package versions are printed in setup.
Small numerical or selection differences can still occur between versions.